In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, Button, VBox, HBox, Output, interactive_output
from pathlib import Path

plots_dir = Path("plots")
plots_dir.mkdir(exist_ok=True)

plot_output = Output()
last_fig = {"fig": None, "variance": 0.05, "theta_peak": 2.0}

theta = np.linspace(0, 4, 1000)

def p_x_given_theta(theta_vals, k=3):
    raw = 1 + np.sin(k * theta_vals)
    Z = np.trapezoid(raw, theta_vals)
    return raw / Z

def plot_posterior_predictive_and_sensitivity(variance, theta_peak):
    with plot_output:
        plot_output.clear_output(wait=True)

        f_theta = p_x_given_theta(theta)

        g_theta = np.exp(-0.5 * (theta - theta_peak) ** 2 / variance)
        g_theta /= np.trapezoid(g_theta, theta)
        integrand = f_theta * g_theta
        posterior_predictive = np.trapezoid(integrand, theta)

        theta_prime_vals = np.linspace(0, 4, 200)
        predictive_vals = []
        for tp in theta_prime_vals:
            g = np.exp(-0.5 * (theta - tp) ** 2 / variance)
            g /= np.trapezoid(g, theta)
            pxX = np.trapezoid(f_theta * g, theta)
            predictive_vals.append(pxX)
        predictive_vals = np.array(predictive_vals)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        axes[0].plot(theta, f_theta, label="p(x | theta)", linewidth=2)
        axes[0].plot(theta, g_theta, label="p(theta | X)", linewidth=2)
        axes[0].fill_between(theta, integrand, alpha=0.3, label="p(x|theta) × p(theta|X)")
        axes[0].set_title("Posterior Predictive Integrand View")
        axes[0].set_xlabel("theta")
        axes[0].set_ylabel("Probability Density")
        axes[0].set_xlim(0, 4)
        axes[0].set_ylim(0, 3.0)
        axes[0].legend()
        axes[0].grid(True)

        axes[1].plot(theta_prime_vals, predictive_vals)
        axes[1].axvline(theta_peak, color="black", linestyle="--", label="theta' (posterior center)")
        axes[1].axhline(
            posterior_predictive,
            color="red",
            linestyle="--",
            label=f"Current p(x|X) = {posterior_predictive:.3f}",
        )
        axes[1].set_title("p(x|X) as Function of Posterior Center")
        axes[1].set_xlabel("theta'")
        axes[1].set_ylabel("Probability Density")
        axes[1].set_xlim(0, 4)
        axes[1].grid(True)
        axes[1].legend()

        plt.tight_layout()
        plt.show()

        last_fig["fig"] = fig
        last_fig["variance"] = variance
        last_fig["theta_peak"] = theta_peak

def save_screenshot(_):
    fig = last_fig["fig"]
    if fig is not None:
        variance = last_fig["variance"]
        theta_peak = last_fig["theta_peak"]
        fname = plots_dir.joinpath(
            f"integral_density/integral_density_var{int(variance*1000):04}_theta{int(theta_peak*1000):04}.png"
        )
        fig.savefig(fname)
        print(f"Saved screenshot → {fname}")

variance_slider = FloatSlider(
    value=0.05,
    min=0.02,
    max=1.0,
    step=0.01,
    description="Posterior Var",
    continuous_update=False,
)
theta_peak_slider = FloatSlider(
    value=2.0,
    min=0.0,
    max=4.0,
    step=0.05,
    description="theta\"",
    continuous_update=False,
)
screenshot_button = Button(description="📸 Save Screenshot")
screenshot_button.on_click(save_screenshot)

interactive = interactive_output(
    plot_posterior_predictive_and_sensitivity,
    {"variance": variance_slider, "theta_peak": theta_peak_slider},
)

ui = VBox([HBox([variance_slider, theta_peak_slider, screenshot_button]), plot_output])

display(ui, interactive)